# LIMS API Examples

This notebook demonstrates how to use the LIMS API to query data from the Google Sheets-synchronized SQLite database.

## Overview

The LIMS API provides a read-only interface to query experimental data that is automatically synchronized from Google Sheets. The API includes functions for:

- Listing available tables
- Querying tables with filters
- Searching for specific records
- Getting table schemas
- Converting results to pandas DataFrames

All queries automatically exclude deleted rows unless explicitly requested.

In [1]:
# Import utilities (use util_simple.py for standalone LIMS API usage)
%run util_simple.py

# For visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("\n✓ Notebook setup complete")

✓ LIMS API loaded successfully

✓ Notebook setup complete


## 1. List Available Tables

First, let's see what tables are available in the LIMS database.

In [2]:
# Get list of all tables
tables = get_lims_tables()

print(f"Found {len(tables)} tables:\n")
for table in tables:
    count = get_table_count(table)
    print(f"  {table}: {count} rows")

Found 12 tables:

  Experiments: 6 rows
  Strains: 1020 rows
  Conditions: 14 rows
  Samples: 121 rows
  Measurements: 1154 rows
  Genes: 3356 rows
  Measurement_types: 10 rows
  DNA_constructs: 4 rows
  Primers: 8 rows
  dgoA_alleles_new: 10 rows
  dgoA_alleles_old: 3 rows
  robotic_mt_samples: 0 rows


## 2. Explore Table Schemas

Let's examine the structure of a table to see what columns are available.

In [3]:
# Get schema for the Experiments table
schema = get_table_schema('Experiments')

print("Experiments table schema:\n")
for column, sql_type in schema.items():
    print(f"  {column}: {sql_type}")

Experiments table schema:

  Database_ID: TEXT
  Name: TEXT
  Type: TEXT
  Protocol: TEXT
  Start_timestamp: TEXT
  Description: TEXT
  Report: TEXT
  Error: TEXT


### Tip: Always Check Column Names

When working with a new table, always check the actual column names first to avoid KeyError exceptions. Column names may differ from what you expect due to:
- Spaces being converted to underscores
- Different naming conventions in the Google Sheet
- Additional suffixes (e.g., `_ID`, `_name`)

Use `get_lims_schema(table_name)` or examine `df.columns` to see available columns.

In [4]:
# Example: Safe way to work with unknown schemas
# The get_safe_columns() function is now available from util_simple.py

# Get a sample
sample_df = query_lims('Samples', limit=1)

# Safely select columns
safe_cols = get_safe_columns(sample_df, ['Name', 'Strain', 'Strain_name', 'Type', 'Condition'])
print(f"Available columns from preferred list: {safe_cols}")
print(f"\nAll columns in Samples table:")
for col in sample_df.columns:
    if not col.startswith('deleted') and not col.startswith('last_synced') and not col.startswith('row_hash'):
        print(f"  - {col}")

Available columns from preferred list: ['Name', 'Strain_name', 'Type', 'Condition']

All columns in Samples table:
  - Database_ID
  - Name
  - Experiment
  - Type
  - Condition
  - Strain_name
  - Transforming_DNA
  - Protocol
  - Parent_sample
  - Replicate_samples
  - Innoculation_timestamp
  - Measurements
  - Notes
  - Error


## 3. Query All Records from a Table

Use the helper function `query_lims()` to get all records as a pandas DataFrame.

In [5]:
# Get all experiments as a DataFrame
experiments_df = query_lims('Experiments')

# Display the DataFrame
print(f"Retrieved {len(experiments_df)} experiments\n")
experiments_df

Retrieved 6 experiments



,Database_ID,Name,Type,Protocol,Start_timestamp,Description,Report,Error,deleted,last_synced,row_hash
0,,ALE1b,robotic ALE,ALE1b sample processing,4/4/2025,ALE1b was designed as a proof-of-principle tes...,ALE1b Report,,0,2025-11-14T23:45:28.127262,12b692454937a8256e7ed82c809b3c9f9c64b7db471f9e...
1,,strain_stocks,strain stock testing,NA,5/3/2024,Mock experiment that serves as collection of a...,,,0,2025-11-14T23:45:28.127721,b4500a63abc27a8d2c9d7616a46926f61759e727d4dd51...
2,,TFMN1,robotic mutant competition assay,TFMN1_protocol,10/8/2025,,,incomplete,0,2025-11-14T23:45:28.127755,729503566015225f4430b0f06cf8f4ad4f608b936498bc...
3,,DGOA_EASy,EASy and EASy isolates,EASy_protocol,4/1/2024,Prototrophic selection of strains with dgoA al...,,,0,2025-11-14T23:45:28.127781,9d1b1cb9ef02ce144c7a245c99711b31933c4dcf847041...
4,,ANL_UGA_prot_1,inter lab comparison,,,Comparison of proteomics results from ADP1 sam...,,incomplete,0,2025-11-14T23:45:28.127804,ee6fe35188ca21ab662e40885c83faceb38edf94cd15aa...
5,,ANL_UGA_prot_2,inter lab comparison,,3/26/2025,Repeat experiment of ANL_UGA_prot_1 after stan...,,incomplete,0,2025-11-14T23:45:28.127831,549ebfba9657ff2af1d1e26aa7a411ec51963eb5eac8bf...


## 4. Query with Filters

Filter records by specific column values.

In [6]:
# Get only robotic ALE experiments
robotic_experiments = query_lims(
    'Experiments',
    filters={'Type': 'robotic ALE'}
)

print(f"Found {len(robotic_experiments)} robotic ALE experiments\n")
robotic_experiments

Found 1 robotic ALE experiments



,Database_ID,Name,Type,Protocol,Start_timestamp,Description,Report,Error,deleted,last_synced,row_hash
0,,ALE1b,robotic ALE,ALE1b sample processing,4/4/2025,ALE1b was designed as a proof-of-principle tes...,ALE1b Report,,0,2025-11-14T23:45:28.127262,12b692454937a8256e7ed82c809b3c9f9c64b7db471f9e...


In [8]:
query_lims(
    'Measurements',
    filters={'Experiment': 'TFMN1', 'Type': 'Short_DNA_reads'}
)

""


## 5. Select Specific Columns

Query only the columns you need to reduce memory usage.

In [ ]:
# Get only specific columns from Experiments
exp_summary = query_lims(
    'Experiments',
    columns=['Name', 'Type', 'Start_timestamp', 'Description']
)

exp_summary

## 6. Limit Results

Use `limit` to get only the first N rows.

In [9]:
# Get first 10 strains
strains_sample = query_lims('Strains', limit=10)

print(f"Showing first {len(strains_sample)} strains out of {get_table_count('Strains')} total\n")
strains_sample

Showing first 10 strains out of 1020 total



,Database_ID,Name,Strain_construction,Genotype,Phenotype,Description,Parent_strain,Supporting_documents,Reference_genome_(GenBank_file),Reference_genome_(SnapGene_file),Reference_genome_(protein_fasta),Reference_genome_created_by,Reference_genome_verified?,Multiple_genotypes_(population?),dgoA_allele,dgoA_Kan_copy_number,Error,deleted,last_synced,row_hash
0,,ADP1,Neidle lab wildtype strain of Acinetobacter ba...,wt,wt,Neidle ADP1 strain has several mutations relat...,None,SRB ADP1 common mutations in our strain,NC_005966-1_Neidle.gbk,ADP1 genome NC_005966.1 with Neidle Lab mutati...,missing,Chanty,TRUE,FALSE,NA,0,,0,2025-11-14T23:45:34.172440,ff019ea1c98c4f2d7ad2a0155cc3370cfae76c592ea384...
1,,ACN2853,Used synthetic bridging fragment (SBF) on pBAC...,"ΔACIAD2330, ΔACIAD1878, dgoA-Ec-KmR inserted b...",Population grows without the aromatic amino ac...,We took ACN2821 and replaced its dgoA* with dg...,ACN2850,List of mutations in ACN2821,population strain - multiple genotypes,population strain - multiple genotypes,missing,NA,NA,TRUE,Ecoli,10-50,,0,2025-11-14T23:45:34.172656,511261fc37518ac9568804701f87134278137c35628b07...
2,,ACN2853_T_dgoA-Best,Transformed with dgoA-Best allele using linear...,"ΔACIAD2330, ΔACIAD1878, dgoA-Ec-KmR inserted b...",Population grows without the aromatic amino ac...,ACN2853 was transformed with DNA encoding the ...,ACN2853,List of mutations in ACN2821,population strain - multiple genotypes,population strain - multiple genotypes,missing,NA,NA,TRUE,"Ecoli, Best",1-50,,0,2025-11-14T23:45:34.172675,9becfb65e580eea5bf631a83efa3c651449e3bdfcede09...
3,,ACN3210,Replaced dgoA-Star with dgoA-Best allele by tr...,"ΔaroF52544, ΔaroG52417, dgoA-Best-KmR inserted...","Grows without the aromatic amino acids, KmR",We took ACN2821 and replaced its dgoA* with ou...,ACN2821,List of mutations in ACN2821,ACN3210.gbk,ACN3210.dna,missing,Natascha,FALSE,FALSE,Best,1,,0,2025-11-14T23:45:34.172691,4b1c7d83f0c1d6191550748269e61e28a6456fad137c3d...
4,,ADP1_ATCC,,wt,wt,ATCC 33305. Multiple differences to NCBI ADP1 ...,,,,,missing,,,,,,incomplete,0,2025-11-14T23:45:34.172704,7e5fd1d9772a071b587f48e43015b8cdea23bab04a6c30...
5,,ACN3560,EASy isolate (single copy of ver genes) from p...,verBACKP-KmR between ACIAD_RS16975-ACIAD_RS044...,Isov+; Ver+,Streak purifed EASy isolate from population AC...,ACN3513,Strain Sheet ACN3560.pdf,,,missing,,,FALSE,NA,,incomplete,0,2025-11-14T23:45:34.172716,840bc752ceec716dcb99a97b22495145326723f8f5df69...
6,,ACN2821,EASy isolate from population ACN2676 (pBAC1414...,"ΔaroF52544, ΔaroG52417, dgoA52586, KmR52586, f...",Grows without the aromatic amino acids,"Streak purified EASy isolate, single copy of d...",ACN2586,List of mutations in ACN2821,,,missing,Chanty,TRUE,FALSE,Star,1,incomplete,0,2025-11-14T23:45:34.172747,c85e3211fa318e846d35ae7fe2aba19b5ce1f88c8707e6...
7,,ACN2586,pBAC1837 linearized (AatII) X ACN2567,"ΔaroF52544, ΔaroG52417, dgoA*-KmR inserted bet...","Grows slowly without the aromatic amino acids,...","ADP1 has two native DAHP synthase genes, ACIAD...",ACN2567,,ACN2586_NSS.gbk,ACN2586_NSS.dna,missing,NA,TRUE,FALSE,Star,1,incomplete,0,2025-11-14T23:45:34.172760,f00a7fe1f9ae6442e7450cf23d49a1b3b3bd1832adaa44...
8,,ACN3246,EASy isolate from population ACN2677 (pBAC1414...,"ΔaroF52544, ΔaroG52417, dgoA52666, KmR52586, t...",Grows without the aromatic amino acids,"Streak purified EASy isolate, multiple copies ...",ACN2670,,,,missing,NA,NA,TRUE,Ecoli,dont know approximate copy number (my best gue...,incomplete,0,2025-11-14T23:45:34.172772,ac42ebbd16501c07e28d3c8555077fb8d861a2ea34bf7b...
9,,ACN3636,EASy isolate from population ACN2853 when DNA ...,"Has ACN2821 mutations + others, WGS data avail...",Grows without the aromatic amino acids,EASy isolate from ACN2853 when dgoA alleles we...,ACN2821,List of mutations in ACN2821,,,missing,Chanty,NA,FALSE,new,1,incomplete,0,2025-11-14T23:45:34.172783,2bf19351537c11cc012b298940562e37d9d6ae13ca0a7e...


## 7. Search for Records

Use the `search_lims()` helper to find records where a column contains specific text (case-insensitive).

In [10]:
# Search for strains containing "ADP1" in the Name column
adp1_strains = search_lims('Strains', 'Name', 'ADP1')

print(f"Found {len(adp1_strains)} strains with 'ADP1' in the name\n")
adp1_strains.head(10)

Found 2 strains with 'ADP1' in the name



,Database_ID,Name,Strain_construction,Genotype,Phenotype,Description,Parent_strain,Supporting_documents,Reference_genome_(GenBank_file),Reference_genome_(SnapGene_file),Reference_genome_(protein_fasta),Reference_genome_created_by,Reference_genome_verified?,Multiple_genotypes_(population?),dgoA_allele,dgoA_Kan_copy_number,Error,deleted,last_synced,row_hash
0,,ADP1,Neidle lab wildtype strain of Acinetobacter ba...,wt,wt,Neidle ADP1 strain has several mutations relat...,None,SRB ADP1 common mutations in our strain,NC_005966-1_Neidle.gbk,ADP1 genome NC_005966.1 with Neidle Lab mutati...,missing,Chanty,TRUE,FALSE,NA,0,,0,2025-11-14T23:45:34.172440,ff019ea1c98c4f2d7ad2a0155cc3370cfae76c592ea384...
1,,ADP1_ATCC,,wt,wt,ATCC 33305. Multiple differences to NCBI ADP1 ...,,,,,missing,,,,,,incomplete,0,2025-11-14T23:45:34.172704,7e5fd1d9772a071b587f48e43015b8cdea23bab04a6c30...


## 8. Working with Samples and Measurements

Let's explore how to query related data across tables. First, let's check what columns are available.

In [11]:
# First, let's see what columns are in the Samples table
samples_schema = get_lims_schema('Samples')

print("Samples table columns:")
for col, sql_type in samples_schema.items():
    if not col.startswith('deleted') and not col.startswith('last_synced') and not col.startswith('row_hash'):
        print(f"  {col}: {sql_type}")

Samples table columns:
  Database_ID: TEXT
  Name: TEXT
  Experiment: TEXT
  Type: TEXT
  Condition: TEXT
  Strain_name: TEXT
  Transforming_DNA: TEXT
  Protocol: TEXT
  Parent_sample: TEXT
  Replicate_samples: TEXT
  Innoculation_timestamp: TEXT
  Measurements: TEXT
  Notes: TEXT
  Error: TEXT


In [12]:
# Get all samples from a specific experiment
ale1b_samples = query_lims(
    'Samples',
    filters={'Experiment': 'ALE1b'}
)

print(f"Found {len(ale1b_samples)} samples from experiment ALE1b\n")

# Show relevant columns (adjust based on actual schema)
if len(ale1b_samples) > 0:
    display_cols = [col for col in ['Name', 'Strain_name', 'Condition', 'Type', 'Notes'] 
                    if col in ale1b_samples.columns]
    ale1b_samples[display_cols].head(10)
else:
    print("No samples found for experiment ALE1b")

Found 18 samples from experiment ALE1b



## 9. Combining Queries with Pandas

Use pandas to merge and analyze data from multiple tables.

In [16]:
# Get measurements for a specific sample
if len(ale1b_samples) > 0:
    sample_name = ale1b_samples.iloc[10]['Name']
    
    measurements = query_lims(
        'Measurements',
        filters={'Sample_ID': sample_name}
    )
    
    if len(measurements) > 0:
        print(f"Measurements for sample {sample_name}:\n")
        # Show relevant columns
        display_cols = [col for col in ['Name', 'Type', 'Timestamp', 'Data', 'Protocol'] 
                        if col in measurements.columns]
        display(measurements[display_cols].head(10))
    else:
        print(f"No measurements found for sample {sample_name}")
        print("\nTrying to find any measurements...")
        all_measurements = query_lims('Measurements', limit=5)
        if len(all_measurements) > 0:
            print(f"\nFound {get_table_count('Measurements')} total measurements")
            display(all_measurements.head())
else:
    print("No samples found to query measurements")

Measurements for sample ALE1b.ACN2853_T_dgoA-Best.pyruvate.1:



,Name,Type,Timestamp,Data,Protocol
0,ALE1b.ACN2853_T_dgoA-Best.pyruvate.1.first.Sho...,Short_DNA_reads,7/14/2025,"E2_S1_L001_R2_001.fastq.gz, E2_S1_L001_R1_001....",
1,ALE1b.ACN2853_T_dgoA-Best.pyruvate.1.last.Shor...,Short_DNA_reads,7/14/2025,"E4_S2_L001_R1_001.fastq.gz, E4_S2_L001_R2_001....",
2,ALE1b.ACN2853_T_dgoA-Best.pyruvate.1.first.Lon...,Long_DNA_reads,7/17/2025,,
3,ALE1b.ACN2853_T_dgoA-Best.pyruvate.1.last.Long...,Long_DNA_reads,7/17/2025,,
4,ALE1b.ACN2853_T_dgoA-Best.pyruvate.1.first.OD_...,OD_series_robot,4/4/2025,ALE1b_robotic_OD,
5,ALE1b.ACN2853_T_dgoA-Best.pyruvate.1.first.OD_...,OD_series_flask,5/6/2025,shake_flask_OD_readings_from_select_ALE1b_robo...,
6,ALE1b.ACN2853_T_dgoA-Best.pyruvate.1.last.OD_s...,OD_series_flask,5/6/2025,shake_flask_OD_readings_from_select_ALE1b_robo...,
7,ALE1b.ACN2853_T_dgoA-Best.pyruvate.1.first.qPCR,qPCR,5/21/2025,Summary of ALE1b qPCR results.xlsx,
8,ALE1b.ACN2853_T_dgoA-Best.pyruvate.1.last.qPCR,qPCR,5/21/2025,Summary of ALE1b qPCR results.xlsx,
9,ALE1b.ACN2853_T_dgoA-Best.pyruvate.1.first.RNAseq,RNAseq,7/30/2025,ALE1b_RNAseq,


## 10. Advanced: Direct API Usage

For more control, you can use the raw LIMS API functions directly.

In [29]:
# Use the raw API for more complex queries
from aisynbiopipeline.limsapi import query_table

# Query with ordering
results = query_table(
    'Genes',
    columns=['Locus_tag', 'Name', 'Function'],
    order_by='Locus_tag',
    order_desc=False,
    limit=20
)

results
# # Convert to DataFrame
# genes_df = pd.DataFrame(results)
# print(f"First 20 genes ordered by locus tag:\n")
# genes_df

[{'"Locus_tag"': 'Locus_tag', '"Name"': 'Name', '"Function"': 'Function'},
 {'"Locus_tag"': 'Locus_tag', '"Name"': 'Name', '"Function"': 'Function'},
 {'"Locus_tag"': 'Locus_tag', '"Name"': 'Name', '"Function"': 'Function'},
 {'"Locus_tag"': 'Locus_tag', '"Name"': 'Name', '"Function"': 'Function'},
 {'"Locus_tag"': 'Locus_tag', '"Name"': 'Name', '"Function"': 'Function'},
 {'"Locus_tag"': 'Locus_tag', '"Name"': 'Name', '"Function"': 'Function'},
 {'"Locus_tag"': 'Locus_tag', '"Name"': 'Name', '"Function"': 'Function'},
 {'"Locus_tag"': 'Locus_tag', '"Name"': 'Name', '"Function"': 'Function'},
 {'"Locus_tag"': 'Locus_tag', '"Name"': 'Name', '"Function"': 'Function'},
 {'"Locus_tag"': 'Locus_tag', '"Name"': 'Name', '"Function"': 'Function'},
 {'"Locus_tag"': 'Locus_tag', '"Name"': 'Name', '"Function"': 'Function'},
 {'"Locus_tag"': 'Locus_tag', '"Name"': 'Name', '"Function"': 'Function'},
 {'"Locus_tag"': 'Locus_tag', '"Name"': 'Name', '"Function"': 'Function'},
 {'"Locus_tag"': 'Locus_t

## 11. Analyzing Measurement Data

Let's do a more complex analysis combining multiple tables.

In [1]:
# Get all measurements
all_measurements = query_lims('Measurements')

# Get measurement types if that table exists
try:
    measurement_types = query_lims('Measurement_types')
    has_types_table = True
except:
    has_types_table = False
    print("Measurement_types table not available")

print(f"Total measurements: {len(all_measurements)}")

# Show measurement type distribution
if 'Type' in all_measurements.columns:
    print(f"\nMeasurement types distribution:")
    print(all_measurements['Type'].value_counts())

# Show a sample of the data
print(f"\nSample of measurement data:")
display_cols = [col for col in ['Sample_ID', 'Name', 'Type', 'Data', 'Timestamp'] 
                if col in all_measurements.columns]
all_measurements[display_cols].head(10)

NameError: name 'query_lims' is not defined

## 12. Summary and Best Practices

### Key Functions

- **`get_lims_tables()`** - List all available tables
- **`query_lims(table, filters, columns, limit)`** - Query a table and return a DataFrame
- **`search_lims(table, column, search_term)`** - Search for records containing text
- **`get_table_schema(table)`** - Get the schema/structure of a table
- **`get_table_count(table)`** - Count rows in a table

### Best Practices

1. **Start with schema exploration**: Use `get_table_schema()` to understand what columns are available
2. **Use filters**: Filter data at the database level rather than in pandas for better performance
3. **Select specific columns**: Only query the columns you need
4. **Use limits for exploration**: When exploring large tables, use `limit` to get a sample first
5. **Leverage pandas**: Use pandas for complex joins, aggregations, and visualizations after querying

### Notes

- All queries automatically exclude deleted rows (soft deletes)
- Column names with spaces or hyphens are converted to underscores in the database
- The database is read-only - no modifications allowed through the API
- Data is automatically synced from Google Sheets every 10 minutes (if daemon is running)